# Visual Fine-Tune Lab — Colab T4 Notebook
**Fine-tune Phi-3.5-Vision with QLoRA on your own document images**

Runtime requirements: **T4 GPU** (Runtime → Change runtime type → T4 GPU)

Pipeline:
1. Install dependencies  
2. Preprocess images with OpenCV  
3. Generate synthetic Q&A dataset via GPT-4o Vision  
4. QLoRA fine-tune Phi-3.5-vision-instruct  
5. Evaluate (BLEU + ROUGE-L + LLM-as-judge)  
6. Log everything to MLflow  
7. Run the full pipeline end-to-end

---
## 1. Environment Setup & Dependencies

In [ ]:
# Check GPU availability
import subprocess, sys

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("No GPU found. Set Runtime → Change runtime type → T4 GPU.")
print(result.stdout[:500])

In [ ]:
%pip install -q \
    "opencv-python-headless>=4.9" \
    "Pillow>=10.3" \
    "openai>=1.30" \
    "transformers>=4.41" \
    "peft>=0.11" \
    "bitsandbytes>=0.43" \
    "accelerate>=0.30" \
    "datasets>=2.19" \
    "torchvision>=0.18" \
    "mlflow>=2.13" \
    "nltk>=3.8" \
    "rouge-score>=0.1.2" \
    "fastapi>=0.111" \
    "python-dotenv>=1.0" \
    "structlog>=24.1" \
    "tqdm>=4.66"

In [ ]:
# Clone repo (skip if working locally)
import os, pathlib

REPO_URL = "https://github.com/tdutra-dev/visual-finetune-lab"  
REPO_DIR = pathlib.Path("/content/visual-finetune-lab")

if not REPO_DIR.exists():
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
os.system("pip install -q -e '.[dev]'")

# Mount Google Drive to persist data / checkpoints
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = pathlib.Path("/content/drive/MyDrive/visual-finetune-lab")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Drive mounted. Working dir: {DRIVE_DIR}")
except ImportError:
    DRIVE_DIR = REPO_DIR
    print("Not running in Colab — using local directory.")

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Option 1: .env file in repo root
load_dotenv(REPO_DIR / ".env")

# Option 2: set secrets directly here (Colab Secrets preferred)
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Validate required keys
required = ["OPENAI_API_KEY"]
missing = [k for k in required if not os.environ.get(k)]
if missing:
    raise ValueError(f"Missing environment variables: {missing}")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db")
BASE_MODEL_ID = os.environ.get("BASE_MODEL_ID", "microsoft/Phi-3.5-vision-instruct")

print(f"OPENAI_API_KEY: {'*' * 8}{OPENAI_API_KEY[-4:]}")
print(f"MLFLOW_TRACKING_URI: {MLFLOW_TRACKING_URI}")
print(f"BASE_MODEL_ID: {BASE_MODEL_ID}")

---
## 2. Image Preprocessing with OpenCV

Upload your images to `data/raw/` on Drive, or use the sample below.

In [ ]:
import pathlib
import numpy as np
from PIL import Image, ImageDraw

# Create sample directory
RAW_DIR = REPO_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Generate a synthetic invoice-like image (no external download needed)
sample_path = RAW_DIR / "sample_invoice.png"
if not sample_path.exists():
    img = Image.new("RGB", (800, 600), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)
    draw.rectangle([40, 40, 760, 100], fill=(30, 80, 160))
    draw.text((50, 55), "INVOICE #2024-001", fill="white")
    draw.text((50, 130), "Date: 2024-01-15", fill=(50, 50, 50))
    draw.text((50, 160), "Customer: Acme Corp", fill=(50, 50, 50))
    draw.line([40, 200, 760, 200], fill=(200, 200, 200), width=2)
    draw.text((50, 220), "Description", fill=(30, 80, 160))
    draw.text((500, 220), "Amount", fill=(30, 80, 160))
    items = [("Consulting Services", "$1,200.00"),
             ("Software License", "$450.00"),
             ("Support -- 3 months", "$300.00")]
    y = 260
    for desc, amt in items:
        draw.text((50, y), desc, fill=(60, 60, 60))
        draw.text((500, y), amt, fill=(60, 60, 60))
        y += 40
    draw.line([40, y+10, 760, y+10], fill=(200, 200, 200), width=2)
    draw.text((400, y+30), "TOTAL: $1,950.00", fill=(30, 80, 160))
    img.save(sample_path)
    print(f"Generated sample invoice -> {sample_path}")

# Copy images from Drive if available
drive_raw = DRIVE_DIR / "raw"
if drive_raw.exists():
    import shutil
    for img_file in drive_raw.glob("*.*"):
        shutil.copy(img_file, RAW_DIR / img_file.name)
    print(f"Copied {len(list(drive_raw.glob('*.*')))} images from Drive")

print(f"Images in {RAW_DIR}: {[p.name for p in RAW_DIR.glob('*.*')]}")


In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR / "src"))

from visual_finetune_lab.preprocessing import ImageProcessor
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

processor = ImageProcessor(max_size=1344)
processed = processor.process_batch(RAW_DIR)
print(f"Processed {len(processed)} images")

# Visualize first image before/after
if processed:
    sample = processed[0]
    # ProcessedImage.path is the processed file; reload original from RAW_DIR
    original = np.array(Image.open(RAW_DIR / sample.path.name))
    result = np.array(Image.fromarray(sample.array[:, :, ::-1]))  # BGR -> RGB

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(original)
    axes[0].set_title("Original", fontsize=13)
    axes[0].axis("off")
    axes[1].imshow(result)
    axes[1].set_title(f"Processed ({sample.width}x{sample.height})", fontsize=13)
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()
    print(f"Detected regions: {sample.regions}")


---
## 3. Synthetic Dataset Generation with GPT-4o Vision

Each preprocessed image is sent to GPT-4o Vision, which generates 5–10 Q&A pairs per image.
The result is saved as a JSONL file compatible with the HuggingFace `datasets` library.

In [ ]:
from visual_finetune_lab.dataset import SyntheticDatasetGenerator
import json

DATASET_DIR = REPO_DIR / "data" / "datasets"
DATASET_DIR.mkdir(parents=True, exist_ok=True)
DATASET_PATH = DATASET_DIR / "dataset.jsonl"

generator = SyntheticDatasetGenerator()

if not DATASET_PATH.exists() or DATASET_PATH.stat().st_size == 0:
    print(f"Generating Q&A pairs for {len(processed)} images...")
    samples = generator.generate(processed)
    generator.save(samples, DATASET_PATH)
    print(f"Saved {len(samples)} samples -> {DATASET_PATH}")
else:
    print(f"Dataset already exists: {DATASET_PATH}")

# Preview first 3 samples
print("
--- Dataset Preview ---")
with open(DATASET_PATH) as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        sample = json.loads(line)
        q = sample["question"]
        a = sample["answer"]
        img = sample["image_path"]
        print(f"[{i+1}] image: {img}")
        print(f"     Q: {q[:80]}")
        print(f"     A: {a[:120]}
")


In [ ]:
# Copy dataset to Drive for persistence
import shutil

drive_datasets = DRIVE_DIR / "datasets"
drive_datasets.mkdir(parents=True, exist_ok=True)
shutil.copy(DATASET_PATH, drive_datasets / "dataset.jsonl")
print(f"Dataset backed up to Drive: {drive_datasets / 'dataset.jsonl'}")

# Count samples
with open(DATASET_PATH) as f:
    n = sum(1 for _ in f)
print(f"Total samples: {n}")

---
## 4. QLoRA Fine-Tuning with PEFT

Loads `Phi-3.5-vision-instruct` in **4-bit NF4** (uses ~6 GB VRAM on T4).  
LoRA adapters are attached only to attention projections — total trainable params ≈ 20M.

| Hyper-parameter | Default | Notes |
|---|---|---|
| `lora_rank` | 16 | Higher = more capacity, more VRAM |
| `lora_alpha` | 32 | Typically 2× rank |
| `num_epochs` | 3 | 3 epochs on ~100 samples takes ~15 min on T4 |
| `batch_size` | 2 | With grad. accum. 4 → effective batch 8 |

In [ ]:
# Training hyper-parameters — edit these freely
LORA_RANK       = 16
LORA_ALPHA      = 32
NUM_EPOCHS      = 3
BATCH_SIZE      = 2
GRAD_ACCUM      = 4
LEARNING_RATE   = 2e-4
CHECKPOINT_DIR  = REPO_DIR / "checkpoints"

In [ ]:
from visual_finetune_lab.training import LoRATrainer, TrainingConfig

config = TrainingConfig(
    base_model_id=BASE_MODEL_ID,
    output_dir=str(CHECKPOINT_DIR),
    num_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    mlflow_experiment="visual-finetune-lab-colab",
)

trainer = LoRATrainer(config)
checkpoint_path = trainer.train(dataset_path=DATASET_PATH)
print(f"\nCheckpoint saved to: {checkpoint_path}")

In [ ]:
# Back up LoRA adapter to Drive
import shutil

drive_ckpt = DRIVE_DIR / "checkpoints" / "best"
drive_ckpt.parent.mkdir(parents=True, exist_ok=True)
if checkpoint_path.exists():
    shutil.copytree(checkpoint_path, drive_ckpt, dirs_exist_ok=True)
    print(f"Adapter backed up to Drive: {drive_ckpt}")
    
    adapter_files = list(drive_ckpt.glob("*"))
    print(f"Files: {[f.name for f in adapter_files]}")

---
## 5. Evaluation: BLEU, ROUGE-L & LLM-as-Judge

Hold out 10% of samples automatically. Metrics:
- **BLEU**: n-gram overlap with reference answers  
- **ROUGE-L**: longest common subsequence ratio  
- **LLM-as-judge**: GPT-4o scores each prediction 1–5 for accuracy

In [ ]:
import json, math

# Build held-out test set (last 10% of dataset)
with open(DATASET_PATH) as f:
    all_samples = [json.loads(l) for l in f]

n_test = max(1, math.ceil(len(all_samples) * 0.1))
test_raw = all_samples[-n_test:]

# Dataset format: {"image_path": ..., "question": ..., "answer": ..., "source_description": ...}
test_samples = [
    {"question": s["question"], "answer": s["answer"], "image_path": s["image_path"]}
    for s in test_raw
]

print(f"Total samples : {len(all_samples)}")
print(f"Train samples : {len(all_samples) - n_test}")
print(f"Eval samples  : {len(test_samples)}")


In [ ]:
from visual_finetune_lab.evaluation import ModelEvaluator

evaluator = ModelEvaluator(checkpoint_path=checkpoint_path)
results = evaluator.evaluate(test_samples)

# Summary table
print(f"{'Question':<50} {'BLEU':>6} {'ROUGE-L':>8} {'LLM':>5}")
print("-" * 75)
for r in results:
    print(f"{r.question[:48]:<50} {r.bleu:>6.3f} {r.rouge_l:>8.3f} {r.llm_judge_score:>5.1f}")

avg_bleu   = sum(r.bleu for r in results) / len(results)
avg_rouge  = sum(r.rouge_l for r in results) / len(results)
avg_judge  = sum(r.llm_judge_score for r in results) / len(results)
print("-" * 75)
print(f"{'AVERAGE':<50} {avg_bleu:>6.3f} {avg_rouge:>8.3f} {avg_judge:>5.1f}")

---
## 6. MLflow Experiment Tracking

Logs are written to `MLFLOW_TRACKING_URI` (SQLite by default).  
For **Databricks Community Edition** set `MLFLOW_TRACKING_URI=databricks` in `.env`.

In [ ]:
import mlflow
import os

os.environ.setdefault("MLFLOW_TRACKING_URI", MLFLOW_TRACKING_URI)
mlflow.set_experiment("visual-finetune-lab-colab")

with mlflow.start_run(run_name=f"qlora-r{LORA_RANK}-e{NUM_EPOCHS}") as run:
    # Log hyper-parameters
    mlflow.log_params({
        "base_model": BASE_MODEL_ID,
        "lora_rank": LORA_RANK,
        "lora_alpha": LORA_ALPHA,
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE * GRAD_ACCUM,
        "learning_rate": LEARNING_RATE,
        "num_train_samples": len(all_samples) - n_test,
        "num_eval_samples": n_test,
    })
    # Log evaluation metrics
    mlflow.log_metrics({
        "bleu": avg_bleu,
        "rouge_l": avg_rouge,
        "llm_judge": avg_judge,
    })
    # Log LoRA adapter as artifact
    if checkpoint_path.exists():
        mlflow.log_artifacts(str(checkpoint_path), artifact_path="lora-adapter")

    RUN_ID = run.info.run_id
    print(f"MLflow run: {RUN_ID}")
    print(f"Experiment: visual-finetune-lab-colab")
    print(f"Metrics — BLEU: {avg_bleu:.3f}  ROUGE-L: {avg_rouge:.3f}  LLM-judge: {avg_judge:.1f}/5")

In [ ]:
# Query the run back from MLflow and display a summary
client = mlflow.tracking.MlflowClient()
run_data = client.get_run(RUN_ID)

print("=" * 55)
print("MLflow Run Summary")
print("=" * 55)
print(f"Run ID   : {RUN_ID}")
print(f"Status   : {run_data.info.status}")
print("\nParameters:")
for k, v in run_data.data.params.items():
    print(f"  {k:<30} {v}")
print("\nMetrics:")
for k, v in run_data.data.metrics.items():
    print(f"  {k:<30} {v:.4f}")

---
## 7. End-to-End Pipeline Execution

Runs the full pipeline — preprocessing → dataset → fine-tune → eval — via `run_pipeline.py`.

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, str(REPO_DIR / "scripts" / "run_pipeline.py"),
    "--images", str(RAW_DIR),
    "--epochs", str(NUM_EPOCHS),
    "--lora-rank", str(LORA_RANK),
]
print("Running:", " ".join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(REPO_DIR))
print(result.stdout)
if result.stderr:
    print("[stderr]", result.stderr[-2000:])
print(f"\nReturn code: {result.returncode}")

In [ ]:
# Verify final artifacts
print("=== Final Artifacts ===")

artifacts = {
    "Dataset (JSONL)":    DATASET_PATH,
    "LoRA adapter":       checkpoint_path / "adapter_model.safetensors",
    "Processor config":   checkpoint_path / "preprocessor_config.json",
    "MLflow DB":          REPO_DIR / "mlflow.db",
}

for label, path in artifacts.items():
    exists = path.exists()
    size = f"{path.stat().st_size / 1024:.1f} KB" if exists else "—"
    status = "✓" if exists else "✗"
    print(f"  {status} {label:<30} {size}")